In [230]:
# ===============================
# STEP 1: Import Libraries & Load Data
# ===============================

# Basic Data Handling
import pandas as pd
import numpy as np

# Visualizations
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling and Evaluation
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [232]:
# # Warnings
# import warnings
# warnings.filterwarnings('ignore')

In [234]:
# Load Dataset
df=pd.read_excel(r"C:\Users\ak32a\Downloads\Anal-ysts\LTF Challenge data with dictionary.xlsx", sheet_name="TrainData")

In [235]:
# View structure
print("Data shape:", df.shape)
df.head()

Data shape: (47970, 105)


,FarmerID,State,REGION,SEX,CITY,Zipcode,DISTRICT,VILLAGE,MARITAL_STATUS,Location,...,Rabi Seasons Agro Ecological Sub Zone in 2020,Rabi Seasons Seasonal average groundwater thickness (cm) in 2020,Rabi Seasons Seasonal average groundwater replenishment rate (cm) in 2020,Night light index,Village score based on socio-economic parameters (Non normalised),Village score based on socio-economic parameters (0 to 100),"Village category based on socio-economic parameters (Good, Average, Poor)",Land Holding Index source (Total Agri Area/ no of people),Road density (Km/ SqKm),Target_Variable/Total Income
0,1002818465057450,MADHYA PRADESH,CENTRAL,M,BARELI,464668,RAISEN,Seoni,M,NaN,...,CENTRAL HIGHLANDS (MALWA AND BUNDELKHAND) HOT...,97.24,19.50,0.95,22.380262,33.527178,Poor,0.773129,0.00,1360000
1,1012300674433870,BIHAR,EAST,M,BANDRA,848125,MUZAFFARPUR,Namapur,M,NaN,...,DECCAN PLATEAU (TELANGANA) AND EASTERN GHATS ...,73.96,16.76,0.97,24.630262,37.173626,Poor,0.454140,0.00,807200
2,1013472263587380,MADHYA PRADESH,CENTRAL,M,MALHARGARH,458556,MANDSAUR,Billaud,M,NaN,...,CENTRAL HIGHLANDS ( MALWA ) GUJARAT PLAIN AND...,90.05,22.44,0.95,19.493313,28.848462,Poor,0.657040,0.00,500000
3,1019525480704050,MAHARASHTRA,WEST,M,RENAPUR,413527,LATUR,Renapur,M,NaN,...,DECCAN PLATU HOT SEMI-ARID ECO-REGION,94.64,21.48,0.98,31.836367,48.852156,Average,0.235615,2.49,558000
4,1021915867444260,MADHYA PRADESH,CENTRAL,F,KHURAI,470117,SAGAR,Singhpur,M,NaN,...,CENTRAL HIGHLANDS (MALWA AND BUNDELKHAND) HOT...,95.90,18.93,0.97,21.327371,31.820817,Poor,0.207264,0.00,800000


In [236]:
# ===============================
# STEP 2: Clean & Standardize Column Names
# ===============================

def clean_column(col_name):
    return (
        col_name.strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace("(", "")
        .replace(")", "")
    )

# Apply to all column names
df.columns = [clean_column(col) for col in df.columns]

# Preview cleaned column names
print("Cleaned Column Names:\n")
print(df.columns.tolist()[:15])  # Show first 15 as a sample


Cleaned Column Names:

['farmerid', 'state', 'region', 'sex', 'city', 'zipcode', 'district', 'village', 'marital_status', 'location', 'address_type', 'ownership', 'no_of_active_loan_in_bureau', 'avg_disbursement_amount_bureau', 'non_agriculture_income']


In [237]:
# ===============================
# STEP 4.5: Save Cleaned Dataset (Optional)
# ===============================

df.to_csv("cleaned_farmer_income_data.csv", index=False)
print("Cleaned dataset saved as cleaned_farmer_income_data.csv ✅")


Cleaned dataset saved as cleaned_farmer_income_data.csv ✅


In [238]:
# ===============================
# STEP 3: Missing Value Analysis
# ===============================

# Total missing values per column
missing_counts = df.isnull().sum()

# Percentage of missing values
missing_percent = (missing_counts / len(df)) * 100

# Combine into a summary table
missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing %': missing_percent
})

# Filter only those with missing values
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing %', ascending=False)

print("Columns with missing values:\n")
missing_df.head(15)


Columns with missing values:



,Missing Count,Missing %
avg_disbursement_amount_bureau,20790,43.339587
location,17030,35.501355
address_type,17030,35.501355
ownership,17030,35.501355
perc_of_house_with_6plus_room,168,0.350219
women_15_19_mothers_or_pregnant_at_time_of_survey,168,0.350219
perc_of_pop_living_in_hh_electricity,168,0.350219
perc_households_with_pucca_house_that_has_more_than_3_rooms,168,0.350219
mat_roof_metal_gi_asbestos_sheets,168,0.350219
perc_of_wall_material_with_burnt_brick,168,0.350219


In [239]:
# there are only 13 columns that have missing values in them.

In [240]:
# ===============================
# STEP 4: Handling Missing Values
# ===============================
df[['location', 'address_type', 'ownership']] = df[['location', 'address_type', 'ownership']].fillna("missing_info")


In [241]:
# Dealing with Avg Disbursement Amt, missing values

# Step 1: Create a binary flag indicating presence of bureau loan data
df['has_loan_bureau_data'] = df['avg_disbursement_amount_bureau'].notna().astype(int)

# Step 2: Replace missing values in Avg_Disbursement_Amount_Bureau with 0
# (Alternatively, you could use -1 to make "missing" more explicit numerically)
df['avg_disbursement_amount_bureau'] = df['avg_disbursement_amount_bureau'].fillna(0)

# Optional: View distribution for sanity check
print(df[['avg_disbursement_amount_bureau', 'has_loan_bureau_data']].head(10))


   avg_disbursement_amount_bureau  has_loan_bureau_data
0                        0.000000                     0
1                    74000.000000                     1
2                   232999.857143                     1
3                        0.000000                     0
4                   138203.000000                     1
5                        0.000000                     0
6                        0.000000                     0
7                        0.000000                     0
8                   100000.000000                     1
9                    66187.000000                     1


In [242]:
# # Count of numeric columns
# num_numeric = (df.select_dtypes(include='number').columns)

# # Count of non-numeric columns
# num_non_numeric = (df.select_dtypes(exclude='number').columns)

# print(f"Numeric columns: {num_numeric}")
# print(f"Non-numeric columns: {num_non_numeric}")


In [243]:
# Count of numeric columns
num_numeric = df.select_dtypes(include='number').columns

# Count of non-numeric columns
num_non_numeric = df.select_dtypes(exclude='number').columns

# ✅ Numeric columns with 0 null values
numeric_no_nulls = [col for col in num_numeric if df[col].isnull().sum() == 0]

print("✅ Numeric columns with 0 null values:")
for col in numeric_no_nulls:
    print(f"- {col}")


✅ Numeric columns with 0 null values:
- farmerid
- zipcode
- no_of_active_loan_in_bureau
- avg_disbursement_amount_bureau
- non_agriculture_income
- k022_proximity_to_nearest_mandi_km
- k022_proximity_to_nearest_railway_km
- ko22_village_score_based_on_socio_economic_parameters_0_to_100
- k022_seasonal_average_rainfall_mm
- r022_seasonal_average_rainfall_mm
- k021_seasonal_average_rainfall_mm
- r021_seasonal_average_rainfall_mm
- r020_seasonal_average_rainfall_mm
- k022_total_geographical_area_in_hectares_
- k022_net_agri_area_in_ha_
- k022_net_agri_area_%_of_total_geog_area_
- kharif_seasons__irrigated_area_in_2022
- kharif_seasons__cropping_density_in_2022
- kharif_seasons__agricultural_performance_in_2022
- kharif_seasons__agricultural_score_in_2022
- kharif_seasons__seasonal_average_groundwater_thickness_cm_in_2022
- kharif_seasons__seasonal_average_groundwater_replenishment_rate_cm_in_2022
- rabi_seasons__season_irrigated_area_in_2022
- rabi_seasons_cropping_density_in_2022
- rabi

In [245]:
# ===============================
# STEP X: Normalize & KNN Impute
# ===============================

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

# 🔹 Define columns to use for KNN input
knn_input_columns = [
    'no_of_active_loan_in_bureau', 'avg_disbursement_amount_bureau', 'non_agriculture_income',
    'k022_proximity_to_nearest_mandi_km', 'k022_proximity_to_nearest_railway_km',
    'ko22_village_score_based_on_socio_economic_parameters_0_to_100',
    'k022_seasonal_average_rainfall_mm', 'r022_seasonal_average_rainfall_mm',
    'k021_seasonal_average_rainfall_mm', 'r021_seasonal_average_rainfall_mm',
    'r020_seasonal_average_rainfall_mm', 'k022_total_geographical_area_in_hectares_',
    'k022_net_agri_area_in_ha_', 'k022_net_agri_area_%_of_total_geog_area_',
    'kharif_seasons__irrigated_area_in_2022', 'kharif_seasons__cropping_density_in_2022',
    'kharif_seasons__agricultural_performance_in_2022', 'kharif_seasons__agricultural_score_in_2022',
    'kharif_seasons__seasonal_average_groundwater_thickness_cm_in_2022',
    'kharif_seasons__seasonal_average_groundwater_replenishment_rate_cm_in_2022',
    'rabi_seasons__season_irrigated_area_in_2022', 'rabi_seasons_cropping_density_in_2022',
    'rabi_seasons_agricultural_performance_in_2022', 'rabi_seasons_agricultural_score_in_2022',
    'rabi_seasons_seasonal_average_groundwater_thickness_cm_in_2022',
    'rabi_seasons_seasonal_average_groundwater_replenishment_rate_cm_in_2022',
    'rabi_seasons_kharif_season_irrigated_area_in_2021', 'rabi_seasons_cropping_density_in_2021',
    'rabi_seasons_agricultural_performance_in_2021', 'rabi_seasons_agricultural_score_in_2021',
    'rabi_seasons_seasonal_average_groundwater_thickness_cm_in_2021',
    'rabi_seasons_seasonal_average_groundwater_replenishment_rate_cm_in_2021',
    'kharif_seasons_kharif_season_irrigated_area_in_2021', 'kharif_seasons_cropping_density_in_2021',
    'kharif_seasons_agricultural_performance_in_2021', 'kharif_seasons_agricultural_score_in_2021',
    'kharif_seasons_seasonal_average_groundwater_thickness_cm_in_2021',
    'kharif_seasons_seasonal_average_groundwater_replenishment_rate_cm_in_2021',
    'kharif_seasons_kharif_season_irrigated_area_in_2020', 'kharif_seasons_cropping_density_in_2020',
    'kharif_seasons_agricultural_performance_in_2020', 'kharif_seasons_agricultural_score_in_2020',
    'kharif_seasons_seasonal_average_groundwater_thickness_cm_in_2020',
    'kharif_seasons_seasonal_average_groundwater_replenishment_rate_cm_in_2020',
    'rabi_seasons_kharif_season_irrigated_area_in_2020', 'rabi_seasons_cropping_density_in_2020',
    'rabi_seasons_agricultural_performance_in_2020', 'rabi_seasons_agricultural_score_in_2020',
    'rabi_seasons_seasonal_average_groundwater_thickness_cm_in_2020',
    'rabi_seasons_seasonal_average_groundwater_replenishment_rate_cm_in_2020',
    'night_light_index', 'village_score_based_on_socio_economic_parameters_non_normalised',
    'village_score_based_on_socio_economic_parameters_0_to_100',
    'land_holding_index_source_total_agri_area__no_of_people',
    'road_density_km__sqkm'
]

# 🔹 Define columns that have missing values and need to be imputed
knn_target_columns = [
    'perc_of_house_with_6plus_room', 'women_15_19_mothers_or_pregnant_at_time_of_survey',
    'perc_of_pop_living_in_hh_electricity', 'perc_households_with_pucca_house_that_has_more_than_3_rooms',
    'mat_roof_metal_gi_asbestos_sheets', 'perc_of_wall_material_with_burnt_brick',
    'households_with_improved_sanitation_facility', 'perc_households_do_not_have_kcc_with_the_credit_limit_of_50k',
    'total_land_for_agriculture'
]

# 🔹 Combine both to apply scaler and imputer
all_knn_columns = knn_input_columns + knn_target_columns

# 🔹 Subset the data
knn_df = df[all_knn_columns].copy()

# 🔹 Normalize
scaler = StandardScaler()
knn_scaled = scaler.fit_transform(knn_df)

# 🔹 Apply KNN imputer
imputer = KNNImputer(n_neighbors=5)
knn_imputed = imputer.fit_transform(knn_scaled)

# 🔹 Convert back to DataFrame
knn_imputed_df = pd.DataFrame(knn_imputed, columns=all_knn_columns)

# 🔹 Update only the target columns in the original df
df[knn_target_columns] = knn_imputed_df[knn_target_columns]

print("✅ KNN imputation completed for selected columns.")


✅ KNN imputation completed for selected columns.


In [246]:
df.to_csv("cleaned_knn_imputed_data.csv", index=False)


In [247]:
# Save the KNN Imputer for Later (e.g., for test data)
import joblib

joblib.dump(scaler, 'scaler_knn_imputer.pkl')
joblib.dump(imputer, 'knn_imputer.pkl')


['knn_imputer.pkl']

In [249]:
print(list(df.keys()))

['farmerid', 'state', 'region', 'sex', 'city', 'zipcode', 'district', 'village', 'marital_status', 'location', 'address_type', 'ownership', 'no_of_active_loan_in_bureau', 'avg_disbursement_amount_bureau', 'non_agriculture_income', 'total_land_for_agriculture', 'k022_village_category_based_on_agri_parameters_good,_average,_poor', 'k022_nearest_mandi_name', 'k022_proximity_to_nearest_mandi_km', 'k022_proximity_to_nearest_railway_km', 'ko22_village_score_based_on_socio_economic_parameters_0_to_100', 'k022_village_category_based_on_socio_economic_parameters_good,_average,_poor', 'k022_seasonal_average_rainfall_mm', 'k022_ambient_temperature_min_&_max', 'r022_village_category_based_on_agri_parameters_good,_average,_poor', 'r022_seasonal_average_rainfall_mm', 'r022_ambient_temperature_min_&_max', 'k021_seasonal_average_rainfall_mm', 'k021_ambient_temperature_min_&_max', 'r021_seasonal_average_rainfall_mm', 'r021_ambient_temperature_min_&_max', 'r020_seasonal_average_rainfall_mm', 'r020_ambie

In [250]:
# Checking for column values similarity to remove similar columns
col_list=[ 'no_of_active_loan_in_bureau', 'avg_disbursement_amount_bureau', 'non_agriculture_income', 'total_land_for_agriculture', 'k022_nearest_mandi_name', 'k022_proximity_to_nearest_mandi_km', 'ko22_village_score_based_on_socio_economic_parameters_0_to_100', 'k022_village_category_based_on_socio_economic_parameters_good,_average,_poor', 'k022_seasonal_average_rainfall_mm', 'k022_ambient_temperature_min_&_max', 'r022_village_category_based_on_agri_parameters_good,_average,_poor', 'r022_seasonal_average_rainfall_mm', 'r022_ambient_temperature_min_&_max', 'k021_seasonal_average_rainfall_mm', 'k021_ambient_temperature_min_&_max', 'r021_seasonal_average_rainfall_mm', 'r021_ambient_temperature_min_&_max', 'r020_seasonal_average_rainfall_mm', 'r020_ambient_temperature_min_&_max', 'k022_total_geographical_area_in_hectares_', 'k022_net_agri_area_in_ha_', 'k022_net_agri_area_%_of_total_geog_area_', 'kharif_seasons__irrigated_area_in_2022', 'kharif_seasons__cropping_density_in_2022', 'kharif_seasons__agricultural_performance_in_2022', 'kharif_seasons__agricultural_score_in_2022', 'kharif_seasons__type_of_soil_in_2022', 'kharif_seasons__type_of_water_bodies_in_hectares_2022', 'kharif_seasons__agro_ecological_sub_zone_in_2022', 'kharif_seasons__seasonal_average_groundwater_thickness_cm_in_2022', 'kharif_seasons__seasonal_average_groundwater_replenishment_rate_cm_in_2022', 'rabi_seasons__season_irrigated_area_in_2022', 'rabi_seasons_cropping_density_in_2022', 'rabi_seasons_agricultural_performance_in_2022', 'rabi_seasons_agricultural_score_in_2022', 'rabi_seasons_type_of_soil_in_2022', 'rabi_seasons_type_of_water_bodies_in_hectares_2022', 'rabi_seasons_agro_ecological_sub_zone_in_2022', 'rabi_seasons_seasonal_average_groundwater_thickness_cm_in_2022', 'rabi_seasons_seasonal_average_groundwater_replenishment_rate_cm_in_2022', 'rabi_seasons_kharif_season_irrigated_area_in_2021', 'rabi_seasons_cropping_density_in_2021', 'rabi_seasons_agricultural_performance_in_2021', 'rabi_seasons_agricultural_score_in_2021', 'rabi_seasons_type_of_soil_in_2021', 'rabi_seasons_type_of_water_bodies_in_hectares_2021', 'rabi_seasons_agro_ecological_sub_zone_in_2021', 'rabi_seasons_seasonal_average_groundwater_thickness_cm_in_2021', 'rabi_seasons_seasonal_average_groundwater_replenishment_rate_cm_in_2021', 'kharif_seasons_kharif_season_irrigated_area_in_2021', 'kharif_seasons_cropping_density_in_2021', 'kharif_seasons_agricultural_performance_in_2021', 'kharif_seasons_agricultural_score_in_2021', 'kharif_seasons_type_of_soil_in_2021', 'kharif_seasons_type_of_water_bodies_in_hectares_2021', 'kharif_seasons_agro_ecological_sub_zone_in_2021', 'kharif_seasons_seasonal_average_groundwater_thickness_cm_in_2021', 'kharif_seasons_seasonal_average_groundwater_replenishment_rate_cm_in_2021', 'kharif_seasons_kharif_season_irrigated_area_in_2020', 'kharif_seasons_cropping_density_in_2020', 'kharif_seasons_agricultural_performance_in_2020', 'kharif_seasons_agricultural_score_in_2020', 'kharif_seasons_type_of_soil_in_2020', 'kharif_seasons_type_of_water_bodies_in_hectares_2020', 'kharif_seasons_agro_ecological_sub_zone_in_2020', 'kharif_seasons_seasonal_average_groundwater_thickness_cm_in_2020', 'kharif_seasons_seasonal_average_groundwater_replenishment_rate_cm_in_2020', 'rabi_seasons_kharif_season_irrigated_area_in_2020', 'rabi_seasons_cropping_density_in_2020', 'rabi_seasons_agricultural_performance_in_2020', 'rabi_seasons_agricultural_score_in_2020', 'rabi_seasons_type_of_soil_in_2020', 'rabi_seasons_type_of_water_bodies_in_hectares_2020', 'rabi_seasons_agro_ecological_sub_zone_in_2020', 'rabi_seasons_seasonal_average_groundwater_thickness_cm_in_2020', 'rabi_seasons_seasonal_average_groundwater_replenishment_rate_cm_in_2020', 'village_score_based_on_socio_economic_parameters_non_normalised', 'village_score_based_on_socio_economic_parameters_0_to_100', 'village_category_based_on_socio_economic_parameters_good,_average,_poor'] 

In [251]:
def check_all_column_similarities(df, col_list=None, threshold=0.95, include_self=False):
    """
    Checks pairwise row-wise similarity of all columns in col_list or all columns in DataFrame.
    Prints and returns column pairs with similarity >= threshold.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The DataFrame to analyze
    col_list : list, optional
        List of column names to check. If None, uses all columns in df
    threshold : float, default 0.95
        Minimum similarity threshold (0.0 to 1.0)
    include_self : bool, default False
        Whether to include self-comparisons (column vs itself)
    
    Returns:
    --------
    dict : Dictionary with column pairs as keys and similarity scores as values
    """
    from itertools import combinations_with_replacement, combinations
    import pandas as pd
    
    # Use all columns if col_list is not provided
    if col_list is None:
        col_list = df.columns.tolist()
    
    # Validate columns exist in DataFrame
    missing_cols = [col for col in col_list if col not in df.columns]
    if missing_cols:
        print(f"⚠️  Warning: These columns don't exist in DataFrame: {missing_cols}")
        col_list = [col for col in col_list if col in df.columns]
    
    if len(col_list) < 2:
        print("❌ Need at least 2 valid columns to compare")
        return {}
    
    results = {}
    total_comparisons = 0
    high_similarity_count = 0
    
    print(f"🔍 Checking similarity between {len(col_list)} columns...")
    print(f"📊 Threshold: {threshold * 100:.1f}%")
    print("-" * 80)
    
    # Choose iteration method based on include_self parameter
    if include_self:
        column_pairs = combinations_with_replacement(col_list, 2)
    else:
        column_pairs = combinations(col_list, 2)
    
    for col1, col2 in column_pairs:
        total_comparisons += 1
        
        # Handle case where both columns are the same (only if include_self=True)
        if col1 == col2:
            similarity = 1.0
        else:
            # Get valid (non-null) mask for both columns
            valid_mask = df[col1].notnull() & df[col2].notnull()
            valid_count = valid_mask.sum()
            
            if valid_count == 0:
                similarity = 0.0
                print(f"⚠️  {col1} vs {col2} → No valid data to compare")
                continue
            else:
                # Calculate similarity only on valid data
                similarity = (df[col1] == df[col2])[valid_mask].mean()
        
        similarity = round(similarity, 4)
        
        # Store and print results that meet threshold
        if similarity >= threshold:
            high_similarity_count += 1
            pair_key = f"{col1} vs {col2}"
            results[pair_key] = similarity
            
            # Enhanced output with more details
            valid_count = (df[col1].notnull() & df[col2].notnull()).sum() if col1 != col2 else len(df)
            total_rows = len(df)
            
            print(f"✅ {col1} vs {col2}")
            print(f"   📈 Similarity: {similarity * 100:.2f}%")
            print(f"   📋 Valid comparisons: {valid_count}/{total_rows} rows")
            print()
    
    # Summary statistics
    print("-" * 80)
    print(f"📊 SUMMARY:")
    print(f"   Total comparisons made: {total_comparisons}")
    print(f"   High similarity pairs found: {high_similarity_count}")
    print(f"   Percentage of high similarity pairs: {(high_similarity_count/total_comparisons)*100:.1f}%")
    
    if results:
        print(f"\n🏆 TOP SIMILAR PAIRS:")
        sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
        for i, (pair, sim) in enumerate(sorted_results[:5], 1):
            print(f"   {i}. {pair}: {sim*100:.2f}%")
    else:
        print(f"❌ No column pairs found with similarity >= {threshold*100:.1f}%")
    
    return results


def analyze_similar_columns_detailed(df, col_list=None, threshold=0.95):
    """
    Extended analysis that also shows sample differences for high similarity pairs.
    """
    results = check_all_column_similarities(df, col_list, threshold)
    
    if not results:
        return results
    
    print("\n" + "="*80)
    print("🔬 DETAILED ANALYSIS OF SIMILAR COLUMNS")
    print("="*80)
    
    for pair, similarity in results.items():
        col1, col2 = pair.split(" vs ")
        
        print(f"\n📋 Analyzing: {col1} vs {col2} (Similarity: {similarity*100:.2f}%)")
        
        # Find differences
        valid_mask = df[col1].notnull() & df[col2].notnull()
        differences_mask = (df[col1] != df[col2]) & valid_mask
        diff_count = differences_mask.sum()
        
        if diff_count > 0:
            print(f"   🔍 Found {diff_count} differences out of {valid_mask.sum()} valid comparisons")
            
            # Show sample differences
            diff_sample = df[differences_mask][[col1, col2]].head(3)
            if not diff_sample.empty:
                print(f"   📝 Sample differences:")
                for idx, row in diff_sample.iterrows():
                    print(f"      Row {idx}: '{row[col1]}' vs '{row[col2]}'")
        else:
            print(f"   ✨ Perfect match! No differences found.")
        
        # Show unique value counts
        print(f"   📊 Unique values: {col1} ({df[col1].nunique()}), {col2} ({df[col2].nunique()})")
    
    return results


# Example usage:
"""
# Basic usage - check all columns in DataFrame
results = check_all_column_similarities(df, threshold=0.95)

# Check specific columns only
specific_cols = ['income', 'revenue', 'profit', 'earnings']
results = check_all_column_similarities(df, col_list=specific_cols, threshold=0.90)

# Include self-comparisons (useful for debugging)
results = check_all_column_similarities(df, threshold=0.95, include_self=True)

# Detailed analysis with sample differences
results = analyze_similar_columns_detailed(df, threshold=0.95)
"""

"\n# Basic usage - check all columns in DataFrame\nresults = check_all_column_similarities(df, threshold=0.95)\n\n# Check specific columns only\nspecific_cols = ['income', 'revenue', 'profit', 'earnings']\nresults = check_all_column_similarities(df, col_list=specific_cols, threshold=0.90)\n\n# Include self-comparisons (useful for debugging)\nresults = check_all_column_similarities(df, threshold=0.95, include_self=True)\n\n# Detailed analysis with sample differences\nresults = analyze_similar_columns_detailed(df, threshold=0.95)\n"

In [252]:
check_all_column_similarities(df,col_list,0.95)

🔍 Checking similarity between 79 columns...
📊 Threshold: 95.0%
--------------------------------------------------------------------------------
✅ ko22_village_score_based_on_socio_economic_parameters_0_to_100 vs village_score_based_on_socio_economic_parameters_0_to_100
   📈 Similarity: 100.00%
   📋 Valid comparisons: 47970/47970 rows

✅ k022_village_category_based_on_socio_economic_parameters_good,_average,_poor vs village_category_based_on_socio_economic_parameters_good,_average,_poor
   📈 Similarity: 100.00%
   📋 Valid comparisons: 47970/47970 rows

✅ kharif_seasons__type_of_soil_in_2022 vs rabi_seasons_type_of_soil_in_2022
   📈 Similarity: 100.00%
   📋 Valid comparisons: 47970/47970 rows

✅ kharif_seasons__type_of_soil_in_2022 vs rabi_seasons_type_of_soil_in_2021
   📈 Similarity: 100.00%
   📋 Valid comparisons: 47970/47970 rows

✅ kharif_seasons__type_of_soil_in_2022 vs kharif_seasons_type_of_soil_in_2021
   📈 Similarity: 100.00%
   📋 Valid comparisons: 47970/47970 rows

✅ kharif_se

{'ko22_village_score_based_on_socio_economic_parameters_0_to_100 vs village_score_based_on_socio_economic_parameters_0_to_100': 1.0,
 'k022_village_category_based_on_socio_economic_parameters_good,_average,_poor vs village_category_based_on_socio_economic_parameters_good,_average,_poor': 1.0,
 'kharif_seasons__type_of_soil_in_2022 vs rabi_seasons_type_of_soil_in_2022': 1.0,
 'kharif_seasons__type_of_soil_in_2022 vs rabi_seasons_type_of_soil_in_2021': 1.0,
 'kharif_seasons__type_of_soil_in_2022 vs kharif_seasons_type_of_soil_in_2021': 1.0,
 'kharif_seasons__type_of_soil_in_2022 vs rabi_seasons_type_of_soil_in_2020': 1.0,
 'kharif_seasons__type_of_water_bodies_in_hectares_2022 vs rabi_seasons_type_of_water_bodies_in_hectares_2022': 1.0,
 'kharif_seasons__type_of_water_bodies_in_hectares_2022 vs rabi_seasons_type_of_water_bodies_in_hectares_2021': 1.0,
 'kharif_seasons__type_of_water_bodies_in_hectares_2022 vs kharif_seasons_type_of_water_bodies_in_hectares_2021': 1.0,
 'kharif_seasons__t

In [253]:
# columns to drop as they have same values throughout

drop_col = ['kharif_seasons_cropping_density_in_2021', 
            'village_score_based_on_socio_economic_parameters_0_to_100', 
            'rabi_seasons_type_of_soil_in_2021', 'kharif_seasons__type_of_soil_in_2022', 
            'rabi_seasons_type_of_soil_in_2022', 'kharif_seasons_type_of_soil_in_2020',
            'rabi_seasons_agro_ecological_sub_zone_in_2020', 'rabi_seasons_type_of_soil_in_2020', 
            'rabi_seasons_type_of_water_bodies_in_hectares_2021', 
            'rabi_seasons_type_of_water_bodies_in_hectares_2022', 
            'rabi_seasons_agro_ecological_sub_zone_in_2021', 
            'rabi_seasons_type_of_water_bodies_in_hectares_2020', 
            'kharif_seasons_kharif_season_irrigated_area_in_2021', 
            'kharif_seasons_type_of_water_bodies_in_hectares_2021',
            'kharif_seasons_type_of_water_bodies_in_hectares_2020',
            'kharif_seasons_agro_ecological_sub_zone_in_2021' ,
            'kharif_seasons_agro_ecological_sub_zone_in_2021' ,
            'rabi_seasons_agro_ecological_sub_zone_in_2022' ]

In [254]:
# In addition to the above list, some columns are just increasing the noise in the data, such as
# zipcode, location, etc and some other are just repeating the data already given in numeric form. 
# So they are to be dropped.

In [255]:
columns_to_be_dropped = ['zipcode','location','city','village',
                         'r022_village_category_based_on_agri_parameters_good,_average,_poor',
                         'k022_village_category_based_on_agri_parameters_good,_average,_poor',
                         'k022_village_category_based_on_socio_economic_parameters_good,_average,_poor',
                         'village_category_based_on_socio_economic_parameters_good,_average,_poor']
drop_col.extend(columns_to_be_dropped)

In [256]:
drop_col

['kharif_seasons_cropping_density_in_2021',
 'village_score_based_on_socio_economic_parameters_0_to_100',
 'rabi_seasons_type_of_soil_in_2021',
 'kharif_seasons__type_of_soil_in_2022',
 'rabi_seasons_type_of_soil_in_2022',
 'kharif_seasons_type_of_soil_in_2020',
 'rabi_seasons_agro_ecological_sub_zone_in_2020',
 'rabi_seasons_type_of_soil_in_2020',
 'rabi_seasons_type_of_water_bodies_in_hectares_2021',
 'rabi_seasons_type_of_water_bodies_in_hectares_2022',
 'rabi_seasons_agro_ecological_sub_zone_in_2021',
 'rabi_seasons_type_of_water_bodies_in_hectares_2020',
 'kharif_seasons_kharif_season_irrigated_area_in_2021',
 'kharif_seasons_type_of_water_bodies_in_hectares_2021',
 'kharif_seasons_type_of_water_bodies_in_hectares_2020',
 'kharif_seasons_agro_ecological_sub_zone_in_2021',
 'kharif_seasons_agro_ecological_sub_zone_in_2021',
 'rabi_seasons_agro_ecological_sub_zone_in_2022',
 'zipcode',
 'location',
 'city',
 'village',
 'r022_village_category_based_on_agri_parameters_good,_average,_

In [257]:
df.drop(columns=drop_col, inplace=True)

In [258]:
df.shape

(47970, 81)

In [260]:
# Now let's deal with water bodies, we are converting them to binary flags

In [261]:
# First, replace missing/None-like values with empty string
df['kharif_seasons__type_of_water_bodies_in_hectares_2022'] = df['kharif_seasons__type_of_water_bodies_in_hectares_2022'].fillna("").astype(str).str.lower()

# Define keywords to look for
keywords = ['river', 'water', 'reservoir', 'riverbank', 'wetland']

# Create binary columns for each keyword
for kw in keywords:
    df[f'near_{kw}'] = df['kharif_seasons__type_of_water_bodies_in_hectares_2022'].apply(lambda x: int(kw in x))

# Optional: Drop the original column
# df.drop(columns=['water_features'], inplace=True)

# Preview
df[[f'near_{kw}' for kw in keywords]].head()


,near_river,near_water,near_reservoir,near_riverbank,near_wetland
0,0,1,0,0,0
1,0,1,0,0,0
2,1,0,0,0,0
3,0,0,0,0,0
4,0,0,0,0,0


In [262]:
df.shape

(47970, 86)

In [263]:
# separating the min and max in temparature columns for a better interpretation

In [264]:
# Replace with actual column names
temp_cols = [
    'k022_ambient_temperature_min_&_max',
    'r022_ambient_temperature_min_&_max',
    'k021_ambient_temperature_min_&_max',
    'r021_ambient_temperature_min_&_max',
    'r020_ambient_temperature_min_&_max'
]

for col in temp_cols:
    df[[f"{col}_min", f"{col}_max"]] = df[col].str.split('/', expand=True).astype(float)


In [265]:
df.drop(columns=temp_cols, inplace=True)


In [266]:
df.shape

(47970, 91)

In [297]:
df.keys()

Index(['farmerid', 'state', 'region', 'sex', 'district', 'marital_status',
       'address_type', 'ownership', 'no_of_active_loan_in_bureau',
       'avg_disbursement_amount_bureau', 'non_agriculture_income',
       'total_land_for_agriculture', 'k022_nearest_mandi_name',
       'k022_proximity_to_nearest_mandi_km',
       'k022_proximity_to_nearest_railway_km',
       'ko22_village_score_based_on_socio_economic_parameters_0_to_100',
       'k022_seasonal_average_rainfall_mm',
       'r022_seasonal_average_rainfall_mm',
       'k021_seasonal_average_rainfall_mm',
       'r021_seasonal_average_rainfall_mm',
       'r020_seasonal_average_rainfall_mm', 'perc_of_house_with_6plus_room',
       'women_15_19_mothers_or_pregnant_at_time_of_survey',
       'perc_of_pop_living_in_hh_electricity',
       'perc_households_with_pucca_house_that_has_more_than_3_rooms',
       'mat_roof_metal_gi_asbestos_sheets',
       'perc_of_wall_material_with_burnt_brick',
       'households_with_improved_sanita

1. Label encode remaining categorical features
2. Do SHAP + Correlation Heatmap
3. Optionally: PCA for visualization (not necessary for modeling)
4. Train XGBoost, CatBoost, LightGBM
5. Evaluate + fine-tune best
6. Optionally: Use VotingRegressor (not classifier since it's a regression task)


In [299]:
# Encoding the Categorical columns, (where caardinality is low, label encoding is applied)
from sklearn.preprocessing import LabelEncoder

label_cols = [
    'region', 'sex', 'marital_status', 'address_type', 
    'ownership', 'kharif_seasons__agro_ecological_sub_zone_in_2022',
    'kharif_seasons_agro_ecological_sub_zone_in_2020'
]

for col in label_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))


In [303]:
# target encoding for high cardinality columns, namely:
# 'district':405, 'k022_nearest_mandi_name':890, 'state':17 

In [313]:
def target_encode_cv(df, col, target, n_splits=5, seed=42):
    df = df.copy()
    df[col + '_target_enc'] = np.nan
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    for train_idx, val_idx in kf.split(df):
        train, val = df.iloc[train_idx], df.iloc[val_idx]
        mapping = train.groupby(col)[target].mean()
        df.loc[df.index[val_idx], col + '_target_enc'] = val[col].map(mapping)
    
    # Safe assignment to avoid chained assignment warning
    global_mean = df[target].mean()
    df[col + '_target_enc'] = df[col + '_target_enc'].fillna(global_mean)
    
    return df


In [315]:
for col in ['district', 'k022_nearest_mandi_name', 'state']:
    df = target_encode_cv(df, col, target='target_variable_total_income')


In [334]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import make_scorer, mean_absolute_error
import numpy as np

# Label-encoded columns (low cardinality)
label_encoded_cols = [
    'region', 'sex', 'marital_status', 'address_type', 
    'ownership', 'kharif_seasons__agro_ecological_sub_zone_in_2022',
    'kharif_seasons_agro_ecological_sub_zone_in_2020'
]

# Target-encoded columns (high cardinality)
target_encoded_cols = [
    'district_target_enc', 
    'k022_nearest_mandi_name_target_enc', 
    'state_target_enc'
]

# Full list of encoded columns
encoded_cols = label_encoded_cols + target_encoded_cols

# # Define features and target
# X = df[encoded_cols]
# y = df['target_variable_total_income']


# Select numerical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Drop target and irrelevant ID
num_cols = [col for col in num_cols if col not in ['farmerid', 'target_variable_total_income']]

# Final feature set
X_full = df[num_cols + encoded_cols]
y = df['target_variable_total_income']


# Define scoring metric
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

# Cross-validation score
scores = cross_val_score(LinearRegression(), X, y, cv=5, scoring=mae_scorer)

print("Cross-validated MAE scores (negative):", scores)
print("Average MAE:", -np.mean(scores))


Cross-validated MAE scores (negative): [-530894.44328382 -470427.07208369 -409507.53141246 -724282.26519149
 -532207.26861302]
Average MAE: 533463.7161168959


In [336]:
scores = cross_val_score(LinearRegression(), X_full, y, cv=5, scoring=mae_scorer)
print("Cross-validated MAE scores (negative):", scores)
print("Average MAE:", -np.mean(scores))


Cross-validated MAE scores (negative): [-327720.30311335 -322754.62859228 -301568.44007297 -445541.7771228
 -387365.93891682]
Average MAE: 356990.2175636447


In [338]:
#  Some engineered Features 
# ===============================
# 1. Relative Income Features
# ===============================
for geo in ['state', 'region', 'district']:
    avg_col = f'avg_income_by_{geo}'
    bin_col = f'above_avg_{geo}_income'

    df[avg_col] = df.groupby(geo)['target_variable_total_income'].transform('mean')
    df[bin_col] = (df['target_variable_total_income'] > df[avg_col]).astype(int)

# ===============================
# 2. Domain-Inspired Features
# ===============================

# Land-based
df['land_per_active_loan'] = df['total_land_for_agriculture'] / (df['no_of_active_loan_in_bureau'] + 1)
df['agri_income_ratio'] = df['target_variable_total_income'] / (df['non_agriculture_income'] + 1)

# Rainfall
rain_cols = [col for col in df.columns if 'rainfall_mm' in col]
df['avg_rainfall'] = df[rain_cols].mean(axis=1)

# Living conditions
df['housing_quality_score'] = (
    df['perc_of_house_with_6plus_room'] +
    df['perc_households_with_pucca_house_that_has_more_than_3_rooms'] +
    df['perc_of_pop_living_in_hh_electricity']
) / 3

# Groundwater
gw_cols = [col for col in df.columns if 'groundwater_thickness' in col]
df['groundwater_variability'] = df[gw_cols].std(axis=1)

# Market access
df['market_access_score'] = 1 / (df['k022_proximity_to_nearest_mandi_km'] + 1)
df['rail_access_score'] = 1 / (df['k022_proximity_to_nearest_railway_km'] + 1)


In [340]:
df.drop(columns=[col + '_target_enc' for col in ['district', 'k022_nearest_mandi_name', 'state']], inplace=True)


In [342]:
# Re-run target encoding on key high-cardinality columns
for col in ['district', 'k022_nearest_mandi_name', 'state']:
    df = target_encode_cv(df, col, target='target_variable_total_income')


In [344]:
target_encoded_cols = [col + '_target_enc' for col in ['district', 'k022_nearest_mandi_name', 'state']]


In [346]:
df.drop(columns=['state', 'district', 'k022_nearest_mandi_name'], inplace=True)

Label Encode Remaining Categorical Features
(i.e. features that weren’t already encoded via label or target enc)

Train Your Models (XGBoost → CatBoost → LightGBM)

Evaluate
(MAE, RMSE, etc.)

Then: SHAP + Correlation Heatmap

SHAP for model interpretability

Correlation Heatmap to detect multi-collinearity, redundant features

(Optional) PCA for visualization or noise reduction (only if dimensionality is hurting model or you want to visualize clusters)

(Optional) Voting Regressor / Ensemble



# Model Training 

In [352]:
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, mean_absolute_error

# Define features and target
X = df.drop(columns=['target_variable_total_income', 'farmerid'])  # Drop target + irrelevant ID
y = df['target_variable_total_income']

# Initialize XGBoost regressor
xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# Cross-validation MAE scorer
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

# Perform cross-validation
scores = cross_val_score(xgb_model, X, y, cv=5, scoring=mae_scorer)

print("Cross-validated MAE scores (negative):", scores)
print("Average MAE:", -np.mean(scores))


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\ak32a\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\ak32a\anaconda3\Lib\site-packages\xgboost\core.py", line 726, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "C:\Users\ak32a\anaconda3\Lib\site-packages\xgboost\sklearn.py", line 1143, in fit
    train_dmatrix, evals = _wrap_evaluation_matrices(
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ak32a\anaconda3\Lib\site-packages\xgboost\sklearn.py", line 603, in _wrap_evaluation_matrices
    train_dmatrix = create_d

In [354]:
non_numeric = df.select_dtypes(exclude=['number', 'bool']).columns
print("Non-numeric columns:", list(non_numeric))

Non-numeric columns: ['kharif_seasons__type_of_water_bodies_in_hectares_2022', 'kharif_seasons_type_of_soil_in_2021']
